In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

In [2]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_np = np.load('Data/X_train_processed.npy')
y_train_np = np.load('Data/y_train_processed.npy')

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32) #Converting numpy arrays into 32-bit tensors to move the math to the GPU for faster training
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor) #Zipping the inputs and targets together and shuffling them in batches of 64 size so the model learns patterns instead of memorizing the order
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"X_train Tensor Shape: {X_train_tensor.shape}")
print(f"y_train Tensor Shape: {y_train_tensor.shape}")

X_train Tensor Shape: torch.Size([53779, 30, 17])
y_train Tensor Shape: torch.Size([53779])


In [3]:
import torch.nn as nn

class PredictiveMaintenanceBiLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=64, num_layers=2):
        super(PredictiveMaintenanceBiLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True) #Bi-LSTM: Reading the 30-cycle history both forwards and backwards
        self.final_decision_layer = nn.Linear(hidden_size * 2, 1) #Creating the funnel to squash the 128 bidirectional thoughts into a single RUL prediction

    def forward(self, x):
        timeline_memory, _ = self.lstm(x)
        final_memory = timeline_memory[:, -1, :] #Slicing the 3D block to grab only the thoughts generated at the very last cycle (cycle 30)
        rul_prediction = self.final_decision_layer(final_memory)
        return rul_prediction

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PredictiveMaintenanceBiLSTM()
model = model.to(device)

print(f"Bi-LSTM Architecture successfully built and loaded onto: {device}")

Bi-LSTM Architecture successfully built and loaded onto: cuda


In [4]:
import torch.optim as optim

mse_calculator = nn.MSELoss() #Setting up the MSE grader and the Adam optimizer (learning rate = 0.001)
adam_optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15

for current_epoch in range(epochs):
    model.train()
    total_epoch_error = 0.0

    for batch_sensor_data, actual_rul in train_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)

        adam_optimizer.zero_grad() #Wiping the calculus memory clean from the previous batch
        predicted_rul = model(batch_sensor_data)
        predicted_rul = predicted_rul.squeeze() #Aligning dimensions for the PyTorch grader
        batch_error = mse_calculator(predicted_rul, actual_rul)
        batch_error.backward() #PyTorch automatically solves the calculus to find the error slopes
        adam_optimizer.step() #Adam adjusts the network's weights based on the slopes

        total_epoch_error += batch_error.item() #Extracting the raw number out of the heavy Tensor to save RAM

    average_epoch_error = total_epoch_error / len(train_loader)
    print(f"Epoch [{current_epoch + 1}/{epochs}] completed. Average Error (MSE): {average_epoch_error:.2f}")

Epoch [1/15] completed. Average Error (MSE): 3931.68
Epoch [2/15] completed. Average Error (MSE): 1758.75
Epoch [3/15] completed. Average Error (MSE): 1722.32
Epoch [4/15] completed. Average Error (MSE): 1722.47
Epoch [5/15] completed. Average Error (MSE): 1723.08
Epoch [6/15] completed. Average Error (MSE): 1723.36
Epoch [7/15] completed. Average Error (MSE): 1722.54
Epoch [8/15] completed. Average Error (MSE): 1722.53
Epoch [9/15] completed. Average Error (MSE): 1723.21
Epoch [10/15] completed. Average Error (MSE): 1722.59
Epoch [11/15] completed. Average Error (MSE): 1723.54
Epoch [12/15] completed. Average Error (MSE): 1722.40
Epoch [13/15] completed. Average Error (MSE): 1723.30
Epoch [14/15] completed. Average Error (MSE): 1723.28
Epoch [15/15] completed. Average Error (MSE): 1722.42


In [5]:
import math

X_test_np = np.load('Data/X_test_processed.npy')
y_test_np = np.load('Data/y_test_processed.npy')

X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) #shuffle=False because we are evaluating, not training

model.eval() #Locking the evaluation pipeline to make sure it doesn't accidentally learn the answers during the test
total_test_error = 0.0

with torch.no_grad(): #Turning off background calculus to save RAM and speed up grading
    for batch_sensor_data, actual_rul in test_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)
        predicted_rul = model(batch_sensor_data).squeeze()
        batch_error = mse_calculator(predicted_rul, actual_rul) 
        total_test_error += batch_error.item()


average_test_mse = total_test_error / len(test_loader)
baseline_rmse = math.sqrt(average_test_mse) #Converting the squared error back into actual engine cycles

print(f"BASELINE RESULTS")
print(f"Test MSE: {average_test_mse:.2f}")
print(f"Test RMSE: {baseline_rmse:.2f} cycles")

 BASELINE RESULTS 
Test MSE: 1969.18
Test RMSE: 44.38 cycles


In [6]:
class AdvancedPredictiveMaintenanceBiLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=128, num_layers=2, dropout_rate=0.3):
        super(AdvancedPredictiveMaintenanceBiLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout_rate) #Increased hidden_size to 128 to give the AI more brain capacity for the FD004 dataset
        self.intermediate_decision_layer = nn.Linear(hidden_size * 2, 64) #Translating 256 LSTM outputs into 64 features before prediction
        self.relu_activation = nn.ReLU() #Allows the network to learn non-linear degradation patterns
        self.dropout_layer = nn.Dropout(dropout_rate) #Heavy dropout to prevent memorization during the long 60-epoch run
        self.final_decision_layer = nn.Linear(64, 1)

    def forward(self, x):
        timeline_memory, _ = self.lstm(x)
        final_memory = timeline_memory[:, -1, :]
        
        out = self.relu_activation(self.intermediate_decision_layer(final_memory))
        out = self.dropout_layer(out)
        rul_prediction = self.final_decision_layer(out)
        
        return rul_prediction

In [7]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

final_epochs = 60
advanced_model = AdvancedPredictiveMaintenanceBiLSTM(hidden_size=128, dropout_rate=0.3).to(device)
adam_optimizer = optim.Adam(advanced_model.parameters(), lr=0.001)
mse_calculator = nn.MSELoss()

scheduler = ReduceLROnPlateau(adam_optimizer, mode='min', patience=5, factor=0.5) #The Automatic Transmission: If the test error stalls for 5 epochs, the learning rate is cut in half. This forces the AI to take smaller, more precise steps as it reaches the bottom of the error valley.

for epoch in range(final_epochs):
    advanced_model.train()
    total_train_error = 0.0
    
    for batch_sensor_data, actual_rul in train_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)
        
        adam_optimizer.zero_grad()
        predicted_rul = advanced_model(batch_sensor_data).squeeze()
        batch_error = mse_calculator(predicted_rul, actual_rul)
        batch_error.backward()
        adam_optimizer.step()
        
        total_train_error += batch_error.item()
        
    average_train_error = total_train_error / len(train_loader)

    advanced_model.eval() #Grading the model on unseen test data every single epoch so the scheduler knows if the model is actually generalizing
    total_test_error = 0.0
    with torch.no_grad():
        for batch_sensor_data, actual_rul in test_loader:
            batch_sensor_data = batch_sensor_data.to(device)
            actual_rul = actual_rul.to(device)
            predicted_rul = advanced_model(batch_sensor_data).squeeze()
            total_test_error += mse_calculator(predicted_rul, actual_rul).item()
    average_test_error = total_test_error / len(test_loader)

    advanced_model.train()
    scheduler.step(average_test_error) #Stepping the scheduler based on the test error (true generalization), not the training error
    current_lr = adam_optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{final_epochs} , Error: {average_train_error:.2f} , LR: {current_lr:.6f}")


advanced_model.eval()
total_test_error = 0.0

with torch.no_grad():
    for batch_sensor_data, actual_rul in test_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)
        
        predicted_rul = advanced_model(batch_sensor_data).squeeze()
        batch_error = mse_calculator(predicted_rul, actual_rul)
        total_test_error += batch_error.item()

final_test_mse = total_test_error / len(test_loader)
final_rmse = math.sqrt(final_test_mse)

print(f"\nADVANCED OPTIMIZED RESULTS")
print(f"Final Test MSE: {final_test_mse:.2f}")
print(f"Final Test RMSE: {final_rmse:.2f} cycles")

torch.save(advanced_model.state_dict(), 'Data/advanced_turbofan_model.pth')

Epoch 01/60 , Error: 2258.67 , LR: 0.001000
Epoch 02/60 , Error: 1576.68 , LR: 0.001000
Epoch 03/60 , Error: 560.85 , LR: 0.001000
Epoch 04/60 , Error: 515.95 , LR: 0.001000
Epoch 05/60 , Error: 458.47 , LR: 0.001000
Epoch 06/60 , Error: 406.81 , LR: 0.001000
Epoch 07/60 , Error: 377.68 , LR: 0.001000
Epoch 08/60 , Error: 369.53 , LR: 0.001000
Epoch 09/60 , Error: 354.54 , LR: 0.001000
Epoch 10/60 , Error: 343.40 , LR: 0.000500
Epoch 11/60 , Error: 331.52 , LR: 0.000500
Epoch 12/60 , Error: 333.24 , LR: 0.000500
Epoch 13/60 , Error: 325.51 , LR: 0.000500
Epoch 14/60 , Error: 321.57 , LR: 0.000500
Epoch 15/60 , Error: 316.59 , LR: 0.000500
Epoch 16/60 , Error: 313.53 , LR: 0.000250
Epoch 17/60 , Error: 310.86 , LR: 0.000250
Epoch 18/60 , Error: 304.07 , LR: 0.000250
Epoch 19/60 , Error: 302.38 , LR: 0.000250
Epoch 20/60 , Error: 301.07 , LR: 0.000250
Epoch 21/60 , Error: 302.69 , LR: 0.000250
Epoch 22/60 , Error: 298.63 , LR: 0.000125
Epoch 23/60 , Error: 294.59 , LR: 0.000125
Epoch 24/